# Customer-Base Audit

Derived from *The Customer-Base Audit: An Excel-Based Companion* (Fader, Hardie, Ross, v1.0).

Quarterly aggregated data is provided by **Madrigal** and represents a 1% sample of data from **Q1 2016 - Q4 2019** (16 quarters) of **70,041 customers**.

Product-dimension analyses (TCBA Ch. 8) are out of scope.

## Imports

In [1]:
import numpy as np
import pandas as pd
import altair as alt
from great_tables import GT, loc, style

## Data

### Long Format

`cust_data_long.csv` — one row per customer × quarter *in which the customer was active*. ~130,789 rows.

| Column | Meaning |
|---|---|
| `CustomerID` | customer key |
| `Cohort` | acquisition quarter, e.g. `y2016 q1`; also `pre y2016` for customers acquired before the observation window |
| `YearQuarter` | `y2016 q1` … `y2019 q4` |
| `NumTrans` | transactions in that quarter |
| `Spend` | revenue in that quarter |
| `Profit` | contribution profit in that quarter |

Derive `Year` from the first 5 characters of `YearQuarter` (`y2016` … `y2019`).

In [2]:
cust_data = pd.read_csv("../../data/madrigal/cust_data_long.csv")
cust_data = (
    cust_data
    .assign(
        Spend=lambda x: (x["Spend"] * 100).round().astype("int64"),
        Profit=lambda x: (x["Profit"] * 100).round().astype("int64"),
    )
    .assign(
        **cust_data["YearQuarter"]
        .str.extract(r"y(\d{4})_q(\d)")
        .rename(columns={0: "Year", 1: "Quarter"})
        .astype({
            "Year": "int32", 
            "Quarter": "int8"
        })
    )
    .drop(columns="YearQuarter")
)

### Wide to Long format

Three files — `cust_by_qtr_trans.csv`, `cust_by_qtr_spend.csv`, `cust_by_qtr_profit.csv` — one row per customer, one column per quarter, plus a `Cohort` column. Mostly zeros/blanks. If you use these, **do not assume the three files share the same CustomerID ordering — verify.**

In this exercise, we will be using the long format data. However, if you only have wide format data, you can create a long format dataframe with the following steps:

```python
from functools import reduce

def wide_to_long(wide_df, value):
    long_data = wide_df.melt(
        id_vars=["CustomerID", "Cohort"], 
        value_vars=wide_df.columns[2:], 
        var_name="YearQuarter", 
        value_name=value
    ).sort_values(
        ["CustomerID", "YearQuarter"]
    )
    
    if value == "NumTrans":
        long_data = long_data.query(
            "NumTrans > 0"
        ).astype({"NumTrans": "int32"})
    
    return long_data.reset_index(drop=True)

trans_wide = pd.read_csv("data/madrigal/cust_by_qtr_trans.csv")
spend_wide = pd.read_csv("data/madrigal/cust_by_qtr_spend.csv")
profit_wide = pd.read_csv("data/madrigal/cust_by_qtr_profit.csv")

cust_data_long = reduce(
    lambda left, right: left.merge(
        right,
        on=["CustomerID", "Cohort", "YearQuarter"],
        how="left",
    ),
    (
        wide_to_long(df, value)
        for df, value in [
            (trans_wide, "NumTrans"),
            (spend_wide, "Spend"),
            (profit_wide, "Profit"),
        ]
    ),
)
```

### Data Conventions

- **Cohort** = customers acquired in a given period (quarter or year).
- **Cohort size** = number of customers whose acquisition period is that period (diagonal of a cohort × period active-customer matrix). Size of the `pre y2016` cohort is **unknown** — exclude it from any %-active or per-cohort-size calculation.
- **AOF** (average order frequency) = total transactions ÷ number of active customers.
- **AOV** (average order value) = total spend ÷ total transactions.
- **Average margin** = total profit ÷ total spend.
- Core multiplicative decomposition of profit:

  ```
  Profit = #customers × AOF × AOV × Margin
         = #customers × (trans/cust) × (spend/trans) × (profit/spend)
  ```

- For cohorts in a period, the decomposition extends to:

  ```
  Cohort profit = cohort size × % cohort active × AOF × AOV × Margin
  Cohort revenue = cohort size × % cohort active × ASPAC
  ASPAC (avg spend per active cohort member) = AOF × AOV
  ```

### Histogram Binning Conventions

- Bins are **half-open on the left**: the `$25–50` bin counts spend in `(25, 50]`. The first bin is inclusive of its lower edge, so a customer with $0 spend falls in the first bin.
- Behavioural distributions are heavily right-skewed (max often 10–100× the mean), so every histogram gets a **right-censoring point** — a terminal `> x` bin.
- Choose bin width from the percentile table, not by rule of thumb. Preferred widths: 1, 2, 5, 10, 20, 25, 50, 100, 200, 250, 500. Too narrow → noisy; too wide → hides the skew.
- Choose the censoring point so the final bin isn't overloaded. If ~5% of customers exceed $578, censoring at $600 puts too much mass in the last bin; $1000 is better.
- Plot **relative frequencies**, not counts, whenever you compare two groups of different size.
- In Python, prefer `np.histogram` with explicit `bins=` edges, or `pd.cut(..., right=True)`. (The Excel `FREQUENCY` warning in the book is not relevant to you.)

## Lens 1 — How do customers differ from one another? (single period)

**Objective:** quantify variability in buying behaviour across customers within one calendar year (e.g. 2019). Answer: the "average customer" is a fiction.

### Working dataset
Filter long data to `Year == y2019`, group by `CustomerID`, sum `NumTrans`, `Spend`, `Profit`. Only customers with ≥1 transaction in 2019 appear.

**Validation targets**

| Metric | Value |
|---|---|
| Active customers, 2019 | 31,855 |
| Total transactions | 60,730 |
| Total spend | ≈ $5,836,712 |
| Total profit | ≈ $2,798,904 |
| Avg transactions / active customer | 1.9 |
| Avg spend / active customer | $183 |
| Avg profit / active customer | $88 |

In [3]:
def yearly_cust_data(df, year):
    
    return (
        df
        .query(f"Year == {year}")
        .groupby("CustomerID", as_index=False)
        .agg(
            NumTrans=("NumTrans", "sum"),
            Spend=("Spend", "sum"),
            Profit=("Profit", "sum"),
        ).assign(
            Spend=lambda x: (x["Spend"] / 100).astype("float32").round(2),
            Profit=lambda x: (x["Profit"] / 100).astype("float32").round(2)
        )
    )

In [4]:
cust_data_2019 = yearly_cust_data(df=cust_data ,year=2019)

summary = {
    "Active Customers": len(cust_data_2019),
    "Total Transactions": cust_data_2019["NumTrans"].sum(),
    "Total Spend": cust_data_2019["Spend"].sum(),
    "Total Profit": cust_data_2019["Profit"].sum(),
    "Avg. Transactions / Active Customer": cust_data_2019["NumTrans"].mean(),
    "Avg. Spend / Active Customer": cust_data_2019["Spend"].mean(),
    "Avg. Profit / Active Customer": cust_data_2019["Profit"].mean(),
}

summary = (
    pd.DataFrame(summary.items(), columns=["Metric", "Value"])
)

(
    GT(summary)
    .tab_header(
        title="2019 Annual Customer Summary",
        subtitle="Customer activity, revenue, and profitability metrics"
    )
    .fmt_number(
        columns="Value",
        rows=[0, 1],
        decimals=0
    ).fmt_currency(
        columns="Value",
        rows=[2, 3],
        decimals=0
    ).fmt_currency(
        columns="Value",
        rows=[4, 5, 6],
        decimals=2
    ).fmt_number(
        columns="Value",
        rows=[4],
        decimals=2
    ).tab_options(
            table_font_size="12px",
            data_row_padding="4px"
    )
)

GT(_tbl_data=                                Metric         Value
0                     Active Customers  3.185500e+04
1                   Total Transactions  6.073000e+04
2                          Total Spend  5.836712e+06
3                         Total Profit  2.798904e+06
4  Avg. Transactions / Active Customer  1.906451e+00
5         Avg. Spend / Active Customer  1.832275e+02
6        Avg. Profit / Active Customer  8.786388e+01, _body=<great_tables._gt_data.Body object at 0x7e5f6ff90ec0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f6ff90ad0>, _spanners=Spanners([]), _heading=Heading(title='2019 Annual Customer Summary', subtitle='Customer activity, revenue, and profitability metrics', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f6ff90d70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f6ff94910>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f6ff90440>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e5f6ff90c20>, <great_tables._gt_data.FormatInfo object at 0x7e5f6ff95090>, <great_tables._gt_data.FormatInfo object at 0x7e5f6ff95590>, <great_tables._gt_data.FormatInfo object at 0x7e5f6ff715b0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8

### Foundational Plots

In [5]:
def customer_descriptives(df, metric):
    s = df[metric]

    mean = s.mean()

    return {
        "count": s.count(),
        "mean": mean,
        "median": s.median(),
        "std": s.std(),
        "min": s.min(),
        "max": s.max(),
        "pct_below_mean": (s < mean).mean(),
        "percentiles": s.quantile([i / 100 for i in range(5, 100, 5)])
    }

def create_percentile_table(df, column, title, subtitle=None, format=None):
    percentile_table = (
        df["percentiles"]
        .reset_index()
        .rename(columns={
            "index": "Percentile",
            column: "Value"
        })
    )

    percentile_table["Percentile"] = (
        percentile_table["Percentile"] * 100
    ).astype(int).astype(str) + "%"


    gt_table = (
        GT(percentile_table)
        .tab_header(
            title=title,
            subtitle=subtitle
        ).tab_options(
            table_font_size="12px",
            data_row_padding="4px"
        )
    )
    
    if format == "currency":
        gt_table = gt_table.fmt_currency(columns="Value", decimals=2)
    elif format == "pct":
        gt_table = gt_table.fmt_percent(columns='Value', decimals=2)
    elif format == "float":
        gt_table = gt_table.fmt_number(columns="Value", decimals=2)
    
    return percentile_table, gt_table

def create_bins_labels(bin_width, max_cutoff, min_cutoff=None):
    if min_cutoff is None:
        min_cutoff = 0
        lower_bins = []
        lower_labels = []
    else:
        lower_bins = [-np.inf]
        lower_labels = [f"<{min_cutoff}"]

    bins = (
        lower_bins
        + list(range(min_cutoff, max_cutoff + bin_width, bin_width))
        + [np.inf]
    )
    
    labels = (
        lower_labels
        + [
            f"{i}-{i+bin_width}"
            for i in range(min_cutoff, max_cutoff, bin_width)
        ]
        + [f"{max_cutoff}+"]
    )
    
    return {"bins": bins, "labels": labels}

def create_distribution(df, column, bins, labels):
    distribution = (
        pd.cut(
            df[column],
            bins=bins,
            labels=labels,
            right=False
        )
        .value_counts()
        .sort_index()
        .reset_index()
        .rename(columns={
            "count": "Customers",
            column: f"{column} Range"
        })
    )

    distribution["Percent"] = (
        distribution["Customers"]
        / distribution["Customers"].sum()
    )

    return distribution


def distribution_barplot(
    distribution,
    column,
    title="Customer Distribution",
    x_title="Range",
    width=800,
    height=400,
):
    return (
        alt.Chart(distribution)
        .mark_bar()
        .encode(
            x=alt.X(
                f"{column} Range:N",
                title=x_title,
                sort=None,
                axis=alt.Axis(
                    labelAngle=-45
                )                
            ),
            y=alt.Y(
                "Percent:Q",
                title="Customers (%)",
                axis=alt.Axis(format="%")
            ),
            tooltip=[
                alt.Tooltip(f"{column} Range:N", title="Range"),
                alt.Tooltip("Customers:Q", title="Customers", format=","),
                alt.Tooltip("Percent:Q", title="Customers (%)", format=".1%"),
            ],
        )
        .properties(
            title=title,
            width=width,
            height=height,
        )
        .configure_axis(
            grid=False
        )
    )

#### Distribution of Spend

Descriptives first (min, max, mean, median, % below mean), then percentiles at 5% intervals (5%…95%), then bin.

**Findings:** max $6,695 ≈ 37× the mean. Mean $183 > median $113. **69% of customers spend below average.** 5th pct $22.20; 10th pct $30.22; the top 5% each spent more than $578.52.

**Bins:** width $25, censor at $1000 → 41 bins. (Width $20 → 51 bins, also defensible.)

**Note:** two customers have exactly $0 spend in 2019; they land in the first bin.

In [6]:
spend_stats = customer_descriptives(cust_data_2019, "Spend")

print(
    f"{'Min spend:':<15}${spend_stats['min']:,.2f}"
    f"\n{'Max spend:':<15}${spend_stats['max']:,.2f}"
    f"\n{'Mean spend:':<15}${spend_stats['mean']:,.2f}"
    f"\n{'Median spend:':<15}${spend_stats['median']:,.2f}"
    f"\n{'% below avg.:':<15}{spend_stats['pct_below_mean']:.1%}"
)

Min spend:     $0.00
Max spend:     $6,695.02
Mean spend:    $183.23
Median spend:  $113.95
% below avg.:  69.2%


In [7]:
print(
    f"Mean spend ${spend_stats['mean']:,.0f} is "
    f"{spend_stats['mean']/spend_stats['median']:.1f}× "
    f"the median spend of ${spend_stats['median']:,.0f}."
)

print(
    f"\n{spend_stats['pct_below_mean']:.0%} of customers "
    f"spent below the average."
)

p = spend_stats["percentiles"]

print(
    f"\nThe bottom 5% of customers spent "
    f"${p.loc[0.05]:,.2f} or less."
)

print(
    f"\nThe bottom 10% of customers spent "
    f"${p.loc[0.10]:,.2f} or less."
)

print(
    f"\nThe top 5% of customers spent more than "
    f"${p.loc[0.95]:,.2f}."
)

Mean spend $183 is 1.6× the median spend of $114.

69% of customers spent below the average.

The bottom 5% of customers spent $22.20 or less.

The bottom 10% of customers spent $30.22 or less.

The top 5% of customers spent more than $578.52.


In [8]:
_, gt_table = create_percentile_table(
    spend_stats, 
    "Spend", 
    title="Customer Spend Distribution",
    subtitle="2019 annual spend percentiles",
    format="currency"
)
gt_table

GT(_tbl_data=   Percentile       Value
0          5%   22.197001
1         10%   30.219999
2         15%   39.599998
3         20%   48.000000
4         25%   57.590000
5         30%   64.790001
6         35%   73.929000
7         40%   86.389999
8         45%   98.906001
9         50%  113.949997
10        55%  126.459999
11        60%  144.000000
12        65%  165.784000
13        70%  188.399994
14        75%  216.370003
15        80%  255.520004
16        85%  311.980011
17        90%  396.951996
18        95%  578.521991, _body=<great_tables._gt_data.Body object at 0x7e5f6efed6e0>, _boxhead=Boxhead([ColInfo(var='Percentile', type=<ColInfoTypeEnum.default: 1>, column_label='Percentile', column_align='right', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f6ff94b90>, _spanners=Spanners([]), _heading=Heading(title='Customer Spend Distribution', subtitle='2019 annual spend percentiles', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f6ef39090>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f6efed480>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f6ef391d0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e5f6efeda70>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), t

In [9]:
distribution_barplot(
    create_distribution(
        cust_data_2019, 
        column="Spend", 
        **create_bins_labels(bin_width=25, max_cutoff=1000)
    ),
    column="Spend",
    title="Customer Spend Distribution (2019)",
    x_title="Annual Spend ($)"
)

alt.Chart(...)

#### Distribution of Profit

Same structure, but with an explicit **`< $0` bin** for loss-making customers.

**Findings:** range –$652 to $3,347. Mean $88 > median $54; again **69% below average**. 5th pct $7.70; 10th pct $12.47; top 5% above $282.45.

**Bins:** width $25, censor at $500, plus the `<0` bin. (Profit mean/median/max run ~45–50% of the corresponding spend figures, which is what motivates the lower cut-off.)


In [10]:
profit_stats = customer_descriptives(cust_data_2019, "Profit")

print(
    f"{'Min profit:':<15}${profit_stats['min']:,.2f}"
    f"\n{'Max profit:':<15}${profit_stats['max']:,.2f}"
    f"\n{'Mean profit:':<15}${profit_stats['mean']:,.2f}"
    f"\n{'Median profit:':<15}${profit_stats['median']:,.2f}"
    f"\n{'% below avg.:':<15}{profit_stats['pct_below_mean']:.1%}"
)

Min profit:    $-651.62
Max profit:    $3,347.09
Mean profit:   $87.86
Median profit: $52.40
% below avg.:  68.7%


In [11]:
print(
    f"The mean profits represent {profit_stats['mean'] / spend_stats['mean'] * 100:,.2f}% of mean spend."
    f"\nThe median profits represent {profit_stats['median'] / spend_stats['median'] * 100:,.2f}% of median spend."
    f"\nThe max profits represent {profit_stats['max'] / spend_stats['max'] * 100:,.2f}% of median spend."
    f"\nThis is approximately 45-50% profit margins for corresponding spend numbers"
)

The mean profits represent 47.95% of mean spend.
The median profits represent 45.99% of median spend.
The max profits represent 49.99% of median spend.
This is approximately 45-50% profit margins for corresponding spend numbers


In [12]:
_, gt_table = create_percentile_table(
    profit_stats, 
    "Profit", 
    title="Customer Profit Distribution",
    subtitle="2019 annual profit percentiles",
    format="currency"
)
gt_table

GT(_tbl_data=   Percentile       Value
0          5%    7.697000
1         10%   12.474000
2         15%   16.681000
3         20%   20.830000
4         25%   24.760000
5         30%   29.110001
6         35%   34.009998
7         40%   39.246001
8         45%   45.689999
9         50%   52.400002
10        55%   60.259998
11        60%   69.353999
12        65%   79.411003
13        70%   90.998000
14        75%  105.959999
15        80%  125.510002
16        85%  153.607001
17        90%  195.881998
18        95%  282.449002, _body=<great_tables._gt_data.Body object at 0x7e5f6ee18490>, _boxhead=Boxhead([ColInfo(var='Percentile', type=<ColInfoTypeEnum.default: 1>, column_label='Percentile', column_align='right', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f6ff94e10>, _spanners=Spanners([]), _heading=Heading(title='Customer Profit Distribution', subtitle='2019 annual profit percentiles', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f6eecd6e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f6ef4f410>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f6ebbc7d0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e5f6ee972f0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'),

In [13]:
distribution_barplot(
    create_distribution(
        cust_data_2019, 
        column="Profit",
        **create_bins_labels(
            bin_width=25, 
            max_cutoff=500, 
            min_cutoff=0
        )
    ),
    column="Profit",
    title="Customer Profit Distribution (2019)",
    x_title="Annual Profit ($)"
)

alt.Chart(...)

#### Distribution of Number of Transactions

Simple value-count of `NumTrans`.

**Findings:** right-skewed (reverse-J). Max = 58 transactions. **63% of customers made exactly one purchase** (20,149 / 31,855), so the mean of 1.9 is misleading.

**Bins:** width 1, censor at `10+`. (Censoring at 5 is too low — 6.8% of customers made ≥5 transactions.) Keep all non-terminal bins **equal width**; do not do 1 / 2–4 / 5–9 / 10+. If you want unequal groupings, use a table, not a histogram.

In [14]:
trans_stats = customer_descriptives(cust_data_2019, "NumTrans")

print(
    f"{'Min trans.:':<15}{trans_stats['min']:,.0f}"
    f"\n{'Max trans.:':<15}{trans_stats['max']:,.0f}"
    f"\n{'Mean trans.:':<15}{trans_stats['mean']:,.2f}"
    f"\n{'Median trans.:':<15}{trans_stats['median']:,.2f}"
    f"\n{'% below avg.:':<15}{trans_stats['pct_below_mean']:.1%}"
)

Min trans.:    1
Max trans.:    58
Mean trans.:   1.91
Median trans.: 1.00
% below avg.:  63.3%


In [15]:
_, gt_table = create_percentile_table(
    trans_stats, 
    "NumTrans", 
    title="Customer Transactions Distribution",
    subtitle="2019 annual transactions percentiles"
)
gt_table

GT(_tbl_data=   Percentile  Value
0          5%    1.0
1         10%    1.0
2         15%    1.0
3         20%    1.0
4         25%    1.0
5         30%    1.0
6         35%    1.0
7         40%    1.0
8         45%    1.0
9         50%    1.0
10        55%    1.0
11        60%    1.0
12        65%    2.0
13        70%    2.0
14        75%    2.0
15        80%    2.0
16        85%    3.0
17        90%    4.0
18        95%    5.0, _body=<great_tables._gt_data.Body object at 0x7e5f6efd3d50>, _boxhead=Boxhead([ColInfo(var='Percentile', type=<ColInfoTypeEnum.default: 1>, column_label='Percentile', column_align='right', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f6eecdba0>, _spanners=Spanners([]), _heading=Heading(title='Customer Transactions Distribution', subtitle='2019 annual transactions percentiles', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f6ee1bdf0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f6ee1bf00>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f6eecdcd0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category=

In [16]:
bins = (list(range(1, 10 + 1, 1)) + [np.inf])
labels = [str(i) for i in range(1, 10, 1)] + ["10+"]

distribution_barplot(
    create_distribution(
        cust_data_2019, 
        column="NumTrans",
        bins=bins,
        labels=labels
    ),
    column="NumTrans",
    title="Customer Transactions Distribution (2019)",
    x_title="Annual Transactions"
)

alt.Chart(...)

#### Distribution of Average Spend per Transaction


Per customer: `avg_spend = Spend / NumTrans`.

**Bins:** width $25, censor at $500. ($20 / $300 also reasonable; the $500 cut-off shows the tail better.)

**Two different "average spend per transaction" numbers**

These are **not** the same quantity and both appear in practice:

- Average spend per customer = **$183**, average transactions per customer = **1.9** → $183 / 1.9 ≈ **$96**
- Mean of each customer's own average spend per transaction = **$99**

The gap isn't an error. They are different estimators.

**AOV (Weighted-average spend per transaction value)**

Start from the ratio of the two per-customer means. The $I$ cancels, leaving total spend over total transactions:

$$
\frac{\frac{1}{I}\sum_{i=1}^{I}\text{spend}_i}{\frac{1}{I}\sum_{i=1}^{I}\text{trans}_i}
=\frac{\sum_{i=1}^{I}\text{spend}_i}{\sum_{i=1}^{I}\text{trans}_i}
$$

Now rewrite the numerator by multiplying and dividing each customer's spend by their transaction count:

$$
=\frac{\sum_{i=1}^{I}\text{trans}_i\times\dfrac{\text{spend}_i}{\text{trans}_i}}{\sum_{i=1}^{I}\text{trans}_i}
=\sum_{i=1}^{I}\left(\frac{\text{trans}_i}{\sum_{j=1}^{I}\text{trans}_j}\right)\frac{\text{spend}_i}{\text{trans}_i}
$$

So it is a weighted average of individual average spend per transaction, where each customer's weight is their **share of total transactions**. Frequent buyers dominate it.

**Unweighted mean of the per-customer average**

$$
\frac{1}{I}\sum_{i=1}^{I}\frac{\text{spend}_i}{\text{trans}_i}
$$

Every customer counts once, regardless of how much they bought. The one-and-done buyers (63% of the base) carry as much weight as the customer with 58 transactions.

**Why it matters**

The two coincide **only** when $\text{trans}_i$ is identical across all $I$ customers. That never happens in a real customer base, so the two numbers will always diverge — and the direction of the gap tells you something. Here $96 < 99$, meaning heavier buyers have *lower* average baskets than light buyers.

Naming convention from the book, worth adopting so you never confuse them again:

| Quantity | Formula | 2019 value |
|---|---|---|
| **AOV** (Average Order Value) | $\sum \text{spend}_i \big/ \sum \text{trans}_i$ | $96 |
| Mean average spend per transaction | $\frac{1}{I}\sum \text{spend}_i / \text{trans}_i$ | $99 |

"AOV" always means the transaction-weighted one. When you see an "average average spend" number in someone else's report, find out which one it is before you act on it.

In [17]:
# per-customer average spend per transaction
cust_data_2019['AvgSpendPerTrans'] = cust_data_2019['Spend'] / cust_data_2019['NumTrans']

avg_spend_stats = customer_descriptives(cust_data_2019, "AvgSpendPerTrans")

print(
    f"{'Min spend:':<15}${avg_spend_stats['min']:,.2f}"
    f"\n{'Max spend:':<15}${avg_spend_stats['max']:,.2f}"
    f"\n{'Mean spend:':<15}${avg_spend_stats['mean']:,.2f}"
    f"\n{'Median spend:':<15}${avg_spend_stats['median']:,.2f}"
    f"\n{'% below avg.:':<15}{avg_spend_stats['pct_below_mean']:.1%}"
)

Min spend:     $0.00
Max spend:     $3,450.98
Mean spend:    $99.01
Median spend:  $77.28
% below avg.:  62.6%


In [18]:
_, gt_table = create_percentile_table(
    avg_spend_stats, 
    "AvgSpendPerTrans", 
    title="Average Spend per Transactions Distribution",
    subtitle=None,
    format="currency"
)
gt_table

GT(_tbl_data=   Percentile       Value
0          5%   19.561000
1         10%   27.540001
2         15%   34.310601
3         20%   40.392000
4         25%   46.218001
5         30%   51.000000
6         35%   58.070297
7         40%   62.998000
8         45%   70.801802
9         50%   77.279999
10        55%   85.190002
11        60%   93.871999
12        65%  104.394997
13        70%  116.697002
14        75%  127.550001
15        80%  143.395505
16        85%  165.298603
17        90%  192.585999
18        95%  241.598999, _body=<great_tables._gt_data.Body object at 0x7e5f6ebe9d30>, _boxhead=Boxhead([ColInfo(var='Percentile', type=<ColInfoTypeEnum.default: 1>, column_label='Percentile', column_align='right', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f6eece060>, _spanners=Spanners([]), _heading=Heading(title='Average Spend per Transactions Distribution', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f6ec40050>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f6ec40650>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f6eece2c0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e5f6ec3c490>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border

In [19]:
distribution_barplot(
    create_distribution(
        cust_data_2019, 
        column="AvgSpendPerTrans",
        **create_bins_labels(
            bin_width=25, 
            max_cutoff=500
        )
    ),
    column="AvgSpendPerTrans",
    title="Average Spend per Transactions Distribution (2019)",
    x_title="Average Spend per Transactions ($)"
)

alt.Chart(...)

#### Average Spend per Transaction, by Transaction Level

Given the variability in both the number of transactions and average spend per transaction across customers, it is natural to ask whether these two quantities are related.

Group customers by transaction count (1, 2, …, 9, `10+` — same bins as §1.3). For each group report: **mean, std dev, min, max, 5th pct, median, 95th pct** of per-customer average spend. Purpose: is average order value related to purchase frequency?

In [20]:
bins = (list(range(1, 10 + 1, 1)) + [np.inf])
labels = [str(i) for i in range(1, 10, 1)] + ["10+"]

cust_data_2019["TransBin"] = pd.cut(
    cust_data_2019["NumTrans"],
    bins=bins,
    labels=labels,
    right=False
)

avg_spend_per_trans = (
    cust_data_2019.groupby("TransBin", as_index=False)
    .agg(
        Mean=("AvgSpendPerTrans", "mean"),
        Std=("AvgSpendPerTrans", "std"),
        Min=("AvgSpendPerTrans", "min"),
        P05=("AvgSpendPerTrans", lambda s: s.quantile(0.05)),
        Median=("AvgSpendPerTrans", "median"),
        P95=("AvgSpendPerTrans", lambda s: s.quantile(0.95)),
        Max=("AvgSpendPerTrans", "max"),
    )
)

(
    GT(avg_spend_per_trans)
    .tab_header(
        title="Analysis of Average Spend per Transaction",
        subtitle="Analysis of average spend per transaction by transaction level"
    ).fmt_currency(
        columns=list(avg_spend_per_trans.columns[1:]),
        decimals=2
    ).tab_options(
            table_font_size="12px",
            data_row_padding="4px"
    )
)

GT(_tbl_data=  TransBin        Mean        Std        Min        P05     Median  \
0        1   99.861561  90.694898   0.000000  18.000000  72.000000   
1        2  100.386581  82.554953   6.880000  23.992250  81.577499   
2        3   99.119952  68.540378   7.763334  26.647667  83.789998   
3        4   96.915875  65.582179   8.820000  27.596250  82.794998   
4        5   91.191284  53.842962   9.788000  27.397600  78.459998   
5        6   91.420917  55.223779  10.958333  25.871334  80.773336   
6        7   86.971100  48.802187  11.281429  27.151357  77.012142   
7        8   88.048181  50.486003  11.250000  29.141250  78.307503   
8        9   81.849074  43.786769  11.311111  24.472666  71.812225   
9      10+   83.371158  43.830915  12.105000  28.944381  74.278184   

          P95          Max  
0  252.444000  3450.979980  
1  231.573000  2544.840088  
2  218.472004   809.533366  
3  206.459999   723.537476  
4  191.167197   400.087988  
5  192.859338   415.456665  
6  180.260147   277.951433  
7  186.086496   339.752502  
8  159.690782   275.942220  
9  167.556258   268.252730  , _body=<great_tables._gt_data.Body object at 0x7e5f6ee68110>, _boxhead=Boxhead([ColInfo(var='TransBin', type=<ColInfoTypeEnum.default: 1>, column_label='TransBin', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='right', column_width=None), ColInfo(var='Std', type=<ColInfoTypeEnum.default: 1>, column_label='Std', column_align='right', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='right', column_width=None), ColInfo(var='P05', type=<ColInfoTypeEnum.default: 1>, column_label='P05', column_align='right', column_width=None), ColInfo(var='Median', type=<ColInfoTypeEnum.default: 1>, column_label='Median', column_align='right', column_width=None), ColInfo(var='P95', type=<ColInfoTypeEnum.default: 1>, column_label='P95', column_align='right', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Max', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f6ee97ad0>, _spanners=Spanners([]), _heading=Heading(title='Analysis of Average Spend per Transaction', subtitle='Analysis of average spend per transaction by transaction level', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f6ec78320>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f6ec78050>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f6ee97d10>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e5f6ec3c7c0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_co

#### Distribution of Average Margin

Per customer: `margin = Profit / Spend`, defined only where `Spend > 0` (two customers have zero spend — exclude them / set NaN, don't zero-fill).

**Caveat:** this is the *overall* margin across all of a customer's 2019 purchasing, not the average of their transaction-level margins. Transaction-level margins aren't recoverable because the source data is aggregated to the quarter.

**Findings:** unlike the other four distributions, margin is **left-skewed**.

**Bins:** width 5%, plus a `< 0%` bin for loss-makers. The book keeps the (85,90], (90,95], (95,100] bins visible even though they're empty — your call.

In [21]:
cust_data_2019 = (
    cust_data_2019
    .assign(Margin=lambda x: x["Profit"] / x["Spend"] * 100)
)

avg_margin_stats = customer_descriptives(cust_data_2019.query("Spend > 0"), "Margin")

print(
    f"{'Min margin:':<15}{avg_margin_stats['min']:,.2f}"
    f"\n{'Max margin:':<15}{avg_margin_stats['max']:,.2f}"
    f"\n{'Mean margin:':<15}{avg_margin_stats['mean']:,.2f}"
    f"\n{'Median margin:':<15}{avg_margin_stats['median']:,.2f}"
    f"\n{'% below avg.:':<15}{avg_margin_stats['pct_below_mean']:.1%}"
)

Min margin:    -366.67
Max margin:    83.33
Mean margin:   46.52
Median margin: 48.38
% below avg.:  40.4%


In [22]:
_, gt_table = create_percentile_table(
    avg_margin_stats, 
    "Margin", 
    title="Average Margin (%) Distribution",
    subtitle=None,
    format="float"
)

gt_table

GT(_tbl_data=   Percentile      Value
0          5%  26.602958
1         10%  34.406146
2         15%  38.347302
3         20%  40.923367
4         25%  42.604195
5         30%  44.054952
6         35%  45.316807
7         40%  46.436002
8         45%  47.455312
9         50%  48.382793
10        55%  49.329439
11        60%  50.302261
12        65%  51.250000
13        70%  52.344319
14        75%  53.419338
15        80%  54.705596
16        85%  56.000000
17        90%  57.574085
18        95%  59.913370, _body=<great_tables._gt_data.Body object at 0x7e5f6ebefc20>, _boxhead=Boxhead([ColInfo(var='Percentile', type=<ColInfoTypeEnum.default: 1>, column_label='Percentile', column_align='right', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f6ec3ce20>, _spanners=Spanners([]), _heading=Heading(title='Average Margin (%) Distribution', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f6ebea350>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f6ebe91d0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f6ec3cd10>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e5f6ec40a50>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=Tru

In [23]:
distribution_barplot(
    create_distribution(
        cust_data_2019.query("Spend > 0"), 
        column="Margin",
        **create_bins_labels(
            bin_width=5, 
            max_cutoff=100,
            min_cutoff=0
        )
    ),
    column="Margin",
    title="Average Margin Distribution (2019)",
    x_title="Margin (%)"
)

alt.Chart(...)

#### Decile analyses

##### Customer decile report — each decile = 10% of *customers*

1. Sort customers by 2019 profit, descending.
2. Rank 1…N.
3. `decile = int(10 * (rank - 1) / N) + 1` — note the `rank - 1`, which is what keeps decile 1 from being off by one customer.
4. Aggregate by decile: customer count, total transactions, total spend, total profit.
5. Then apply the profit decomposition to produce the report columns:

| Column | Formula |
|---|---|
| % of customers | decile customers / total customers |
| % of transactions | decile trans / total trans |
| % of revenue | decile spend / total spend |
| % of profit | decile profit / total profit |
| Avg spend per customer | decile spend / decile customers |
| Avg profit per customer | decile profit / decile customers |
| AOF | decile trans / decile customers |
| AOV | decile spend / decile trans |
| Avg margin | decile profit / decile spend |

The bottom row (totals) gives the firm-level AOF/AOV/margin.

```mermaid
flowchart LR
    P["Profit"] --> X1(("×"))
    X1 --> NC["# Customers"]
    X1 --> APC["Average profit<br/>per customer"]

    APC --> X2(("×"))
    X2 --> AM["Average<br/>margin"]
    X2 --> ASC["Average spend<br/>per customer"]

    ASC --> X3(("×"))
    X3 --> AOF["Average order<br/>frequency (AOF)"]
    X3 --> AOV["Average order<br/>value (AOV)"]

    classDef box fill:#ffffff,stroke:#333,stroke-width:1.5px,color:#000
    classDef op fill:#333,stroke:#333,color:#fff
    class P,NC,APC,AM,ASC,AOF,AOV box
    class X1,X2,X3 op
```

In [24]:
cust_data_2019['ProfitRank'] = (
    pd.qcut(
        cust_data_2019['Profit'].rank(method='first', ascending=False
    ), q=10, labels=False) + 1
)

profit_rank = (
    cust_data_2019
    .groupby("ProfitRank", as_index=False)
    .agg(
        Customers=("CustomerID", "count"),
        Transactions=("NumTrans", "sum"),
        Spend=("Spend", "sum"),
        Profit=("Profit", "sum"),
    )
    .assign(
        PctCust=lambda x: x["Customers"] / x["Customers"].sum(),
        PctTrans=lambda x: x["Transactions"] / x["Transactions"].sum(),
        PctSpend=lambda x: x["Spend"] / x["Spend"].sum(),
        PctProfit=lambda x: x["Profit"] / x["Profit"].sum(),
        AvgSpendCust=lambda x: x["Spend"] / x["Customers"],
        AvgProfitCust=lambda x: x["Profit"] / x["Customers"],
        AOF=lambda x: x["Transactions"] / x["Customers"],
        AOV=lambda x: x["Spend"] / x["Transactions"],
        AvgMargin=lambda x: x["Profit"] / x["Spend"],
    ).drop(
        columns=["Customers", "Transactions", "Spend", "Profit"]
    )
)

fields = ["Decile", "% Cust.", "% Trans.", "% Spend", "% Profit", "Avg Spend/Cust.", "Avg. Profit/Cust.", "AOF", "AOV", "Avg. Margin"]
profit_rank = profit_rank.rename(columns={old: new for old, new in zip(profit_rank.columns, fields)})

(
    GT(profit_rank)
    .tab_header(
        title="Customer Decile Report"
    ).fmt_percent(
        columns=fields[1:5]+[fields[-1]], 
        decimals=1
    ).fmt_currency(
        columns=fields[5:7]+[fields[8]]
    ).fmt_number(
        columns=fields[7]
    ).tab_options(
            table_font_size="12px",
            data_row_padding="4px"
    )
)

GT(_tbl_data=   Decile   % Cust.  % Trans.   % Spend  % Profit  Avg Spend/Cust.  \
0       1  0.100016  0.266030  0.382369  0.397190       700.495763   
1       2  0.099984  0.139272  0.171904  0.176880       315.025334   
2       3  0.100016  0.106735  0.118863  0.121452       217.755787   
3       4  0.099984  0.089379  0.090174  0.090719       165.249941   
4       5  0.100016  0.081113  0.070586  0.068927       129.312431   
5       6  0.099984  0.071793  0.054100  0.051998        99.141856   
6       7  0.099984  0.066343  0.041414  0.038743        75.893686   
7       8  0.100016  0.062095  0.031342  0.028324        57.417628   
8       9  0.099984  0.058604  0.022523  0.018962        41.275216   
9      10  0.100016  0.058637  0.016724  0.006804        30.638541   

   Avg. Profit/Cust.       AOF         AOV  Avg. Margin  
0         348.931929  5.070935  138.139360     0.498121  
1         155.438255  2.655573  118.628008     0.493415  
2         106.695788  2.034526  107.030228     0.489979  
3          79.721949  1.704239   96.964087     0.482433  
4          60.552647  1.546139   83.635689     0.468266  
5          45.694211  1.368917   72.423581     0.460897  
6          34.046767  1.264992   59.995381     0.448611  
7          24.882790  1.183616   48.510359     0.433365  
8          16.663074  1.117425   36.937781     0.403707  
9           5.977492  1.117702   27.412073     0.195097  , _body=<great_tables._gt_data.Body object at 0x7e5f6ec1e7b0>, _boxhead=Boxhead([ColInfo(var='Decile', type=<ColInfoTypeEnum.default: 1>, column_label='Decile', column_align='right', column_width=None), ColInfo(var='% Cust.', type=<ColInfoTypeEnum.default: 1>, column_label='% Cust.', column_align='right', column_width=None), ColInfo(var='% Trans.', type=<ColInfoTypeEnum.default: 1>, column_label='% Trans.', column_align='right', column_width=None), ColInfo(var='% Spend', type=<ColInfoTypeEnum.default: 1>, column_label='% Spend', column_align='right', column_width=None), ColInfo(var='% Profit', type=<ColInfoTypeEnum.default: 1>, column_label='% Profit', column_align='right', column_width=None), ColInfo(var='Avg Spend/Cust.', type=<ColInfoTypeEnum.default: 1>, column_label='Avg Spend/Cust.', column_align='right', column_width=None), ColInfo(var='Avg. Profit/Cust.', type=<ColInfoTypeEnum.default: 1>, column_label='Avg. Profit/Cust.', column_align='right', column_width=None), ColInfo(var='AOF', type=<ColInfoTypeEnum.default: 1>, column_label='AOF', column_align='right', column_width=None), ColInfo(var='AOV', type=<ColInfoTypeEnum.default: 1>, column_label='AOV', column_align='right', column_width=None), ColInfo(var='Avg. Margin', type=<ColInfoTypeEnum.default: 1>, column_label='Avg. Margin', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f6ec3cc00>, _spanners=Spanners([]), _heading=Heading(title='Customer Decile Report', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f6ec98600>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f6ee699d0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f6ec3d040>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e5f6ec41750>, <great_tables._gt_data.FormatInfo object at 0x7e5f6ec0cc80>, <great_tables._gt_data.FormatInfo object at 0x7e5f6ec0ff20>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_col

#### Profit decile report — each decile = 10% of *profit*

1. Sort by profit descending, compute **cumulative profit**.
2. Decile boundaries in cumulative-profit space: `k × (total_profit / 10)` for k = 1…9.
3. For each boundary, find the individual customer profit value at the point the cumulative series first crosses it. These become the profit cut-offs.
4. Assign each customer to a decile by comparing their own profit against those cut-offs.
5. Aggregate and apply the same decomposition table as above.

In [25]:
def decile_labels(df, value_col="Profit", n=10):
    d = df.sort_values(value_col, ascending=False, kind="stable").reset_index(drop=True)

    cents = np.round(d[value_col].to_numpy() * 100).astype(np.int64)
    cum   = cents.cumsum()
    total = cum[-1]

    thresholds = np.arange(1, n) * total / n
    
    boundaries = cents[np.searchsorted(cum, thresholds, side="right")]
    # Profit cutoff
    d["ProfitDecile"] = (n - np.searchsorted(boundaries[::-1], cents, side="left")).astype("int8")
    return d

In [26]:
cust_profit = decile_labels(cust_data_2019, "Profit")

profit_rank = (
    cust_profit
    .groupby("ProfitDecile", as_index=False)
    .agg(
        Customers=("CustomerID", "count"),
        Transactions=("NumTrans", "sum"),
        Spend=("Spend", "sum"),
        Profit=("Profit", "sum"),
    )
    .assign(
        PctCust=lambda x: x["Customers"] / x["Customers"].sum(),
        PctTrans=lambda x: x["Transactions"] / x["Transactions"].sum(),
        PctSpend=lambda x: x["Spend"] / x["Spend"].sum(),
        PctProfit=lambda x: x["Profit"] / x["Profit"].sum(),
        AvgSpendCust=lambda x: x["Spend"] / x["Customers"],
        AvgProfitCust=lambda x: x["Profit"] / x["Customers"],
        AOF=lambda x: x["Transactions"] / x["Customers"],
        AOV=lambda x: x["Spend"] / x["Transactions"],
        AvgMargin=lambda x: x["Profit"] / x["Spend"],
    )
    .drop(
        columns=["Customers", "Transactions", "Spend", "Profit"]
    )
)

fields = ["Decile", "% Cust.", "% Trans.", "% Spend", "% Profit", "Avg Spend/Cust.", "Avg. Profit/Cust.", "AOF", "AOV", "Avg. Margin"]
profit_rank = profit_rank.rename(columns={old: new for old, new in zip(profit_rank.columns, fields)})

(
    GT(profit_rank)
    .tab_header(
        title="Profit Decile Report"
    ).fmt_percent(
        columns=fields[1:5]+[fields[-1]], 
        decimals=2
    ).fmt_currency(
        columns=fields[5:7]+[fields[8]]
    ).fmt_number(
        columns=fields[7]
    ).tab_options(
            table_font_size="12px",
            data_row_padding="4px"
    )
)

GT(_tbl_data=   Decile   % Cust.  % Trans.   % Spend  % Profit  Avg Spend/Cust.  \
0       1  0.010799  0.057517  0.095830  0.099960      1625.957122   
1       2  0.020907  0.065997  0.096208  0.099932       843.148742   
2       3  0.029760  0.070986  0.096586  0.100033       594.666007   
3       4  0.039805  0.073753  0.096400  0.100055       443.736790   
4       5  0.051389  0.077392  0.097406  0.099966       347.299633   
5       6  0.066081  0.081245  0.097205  0.100004       269.529602   
6       7  0.085575  0.089264  0.097810  0.100000       209.422689   
7       8  0.113514  0.100231  0.099828  0.100035       161.135267   
8       9  0.166410  0.127301  0.103235  0.099973       113.667516   
9      10  0.415759  0.256315  0.119494  0.100042        52.661748   

   Avg. Profit/Cust.        AOF         AOV  Avg. Margin  
0         813.307776  10.154070  160.128614     0.500202  
1         419.970017   6.018018  140.104058     0.498097  
2         295.339728   4.547468  130.768586     0.496648  
3         220.855580   3.532334  125.621400     0.497718  
4         170.919804   2.871106  120.963723     0.492139  
5         132.969507   2.343943  114.989828     0.493339  
6         102.674661   1.988628  105.310137     0.490275  
7          77.430085   1.683352   95.722873     0.480528  
8          52.785406   1.458404   77.939659     0.464384  
9          21.142296   1.175325   44.806128     0.401473  , _body=<great_tables._gt_data.Body object at 0x7e5f6ec8e120>, _boxhead=Boxhead([ColInfo(var='Decile', type=<ColInfoTypeEnum.default: 1>, column_label='Decile', column_align='right', column_width=None), ColInfo(var='% Cust.', type=<ColInfoTypeEnum.default: 1>, column_label='% Cust.', column_align='right', column_width=None), ColInfo(var='% Trans.', type=<ColInfoTypeEnum.default: 1>, column_label='% Trans.', column_align='right', column_width=None), ColInfo(var='% Spend', type=<ColInfoTypeEnum.default: 1>, column_label='% Spend', column_align='right', column_width=None), ColInfo(var='% Profit', type=<ColInfoTypeEnum.default: 1>, column_label='% Profit', column_align='right', column_width=None), ColInfo(var='Avg Spend/Cust.', type=<ColInfoTypeEnum.default: 1>, column_label='Avg Spend/Cust.', column_align='right', column_width=None), ColInfo(var='Avg. Profit/Cust.', type=<ColInfoTypeEnum.default: 1>, column_label='Avg. Profit/Cust.', column_align='right', column_width=None), ColInfo(var='AOF', type=<ColInfoTypeEnum.default: 1>, column_label='AOF', column_align='right', column_width=None), ColInfo(var='AOV', type=<ColInfoTypeEnum.default: 1>, column_label='AOV', column_align='right', column_width=None), ColInfo(var='Avg. Margin', type=<ColInfoTypeEnum.default: 1>, column_label='Avg. Margin', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f6ec42c50>, _spanners=Spanners([]), _heading=Heading(title='Profit Decile Report', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f6ee6a2d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f6ebef8b0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f6ec42950>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e5f6ebeadd0>, <great_tables._gt_data.FormatInfo object at 0x7e5f6ebeacf0>, <great_tables._gt_data.FormatInfo object at 0x7e5f6ec9b860>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_backg

**Watch out:** cumulative profit is **not monotonic**. It peaks at **$2,802,771.95** and then *declines* to the total of **$2,798,903.68**, because **263 customers are loss-making**. Decile 1 cut-off lands at ≈ **$545.53**, decile 2 at ≈ **$345.25**. Loss-makers all end up in decile 10.

*Variant worth building:* pull the loss-makers out as a separate 11th group rather than burying them in decile 10.

*Variant worth building:* if you don't have cost/profit data, run the same decile analysis on **revenue**.

## Lens 2 — What changed between two periods? (2018 vs 2019)

**Objective:** decompose year-on-year change in firm performance into changes in customer behaviour.

### Working dataset

One row per customer active in **either** 2018 or 2019 (outer join of the two annual aggregates, zero-filled): `trans_2018, spend_2018, profit_2018, trans_2019, spend_2019, profit_2019`. **48,238 customers.**

Add a `years` status flag:
- `both` if active in both years
- `2018 only`
- `2019 only`

In [40]:
cust_data_2018 = yearly_cust_data(cust_data, 2018)
cust_data_2019 = yearly_cust_data(cust_data, 2019)

cust_2018_2019 = (
    cust_data_2018.merge(
        cust_data_2019,
        on="CustomerID",
        how="outer",
        suffixes=("_2018", "_2019"),
        indicator=True,
    )
    .assign(
        Status=lambda df: df["_merge"].map({
            "both": "both",
            "left_only": "2018 only",
            "right_only": "2019 only",
        })
    )
    .drop(columns="_merge")
)

### Headline

| | 2018 | 2019 | Δ |
|---|---|---|---|
| Revenue | $4,815,502 | $5,836,712 | +21% |
| Profit | $2,290,343 | $2,798,904 | +22% |
| Active customers | 26,254 | 31,855 | +21% |

In [67]:
summary = pd.concat(
    [
        (
            cust_2018_2019.filter(like="_2018")
            .sum(numeric_only=True)
            .rename(lambda c: c.removesuffix("_2018"))
            .rename("2018")
        ), 
        (
            cust_2018_2019.filter(like="_2019")
            .sum(numeric_only=True)
            .rename(lambda c: c.removesuffix("_2019"))
            .rename("2019")
        )
    ],
    axis=1,
)

summary["Δ"] = (summary["2019"] - summary["2018"]) / summary["2018"]

active_2018 = cust_data_2018["CustomerID"].nunique()
active_2019 = cust_data_2019["CustomerID"].nunique()

summary.loc["Active customers"] = [
    active_2018,
    active_2019,
    (active_2019 - active_2018) / active_2018,
]

summary = summary.drop(index="NumTrans").reset_index(names="")

(
    GT(summary)
    .tab_header(
        title="Spend, Profit, & Active Customer YoY Summary"
    ).fmt_percent(
        columns=["Δ"], 
        decimals=1
    ).fmt_currency(
        columns=["2018", "2019"],
        decimals=0
    ).fmt_number(
         columns=["2018", "2019"],
         rows=[2],
         decimals=0
    ).tab_options(
            table_font_size="12px",
            data_row_padding="4px"
    )
)

GT(_tbl_data=                           2018       2019         Δ
0             Spend  4815502.00  5836712.0  0.212067
1            Profit  2290342.75  2798903.5  0.222046
2  Active customers    26254.00    31855.0  0.213339, _body=<great_tables._gt_data.Body object at 0x7e5f4baada90>, _boxhead=Boxhead([ColInfo(var='', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None), ColInfo(var='2018', type=<ColInfoTypeEnum.default: 1>, column_label='2018', column_align='right', column_width=None), ColInfo(var='2019', type=<ColInfoTypeEnum.default: 1>, column_label='2019', column_align='right', column_width=None), ColInfo(var='Δ', type=<ColInfoTypeEnum.default: 1>, column_label='Δ', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5f4b6287a0>, _spanners=Spanners([]), _heading=Heading(title='Spend, Profit, & Active Customer YoY Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5f4b66ae10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5f4b66b410>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5f4b6288c0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e5f4b66b9b0>, <great_tables._gt_data.FormatInfo object at 0x7e5f4b66b170>, <great_tables._gt_data.FormatInfo object at 0x7e5f4b66af90>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value

### Overlaid distributions

Re-run each Lens 1 distribution for 2018 and 2019 on the same axes, using **identical bin edges** and **relative frequencies** (the customer counts differ, so raw counts are not comparable).

**Finding:** the spend distribution for 2018 actives is very close to that for 2019 actives — the shape of customer heterogeneity is stable; the customer *count* is what moved. (Repeat for profit, transactions, avg spend per transaction, margin.)

### 2.3 Customer overlap
| Group | Customers |
|---|---|
| Active 2018 | 26,254 |
| Active 2019 | 31,855 |
| Active both years | 9,871 |
| 2018 only (lapsed) | 16,383 |
| 2019 only (new/reactivated) | 21,984 |

Repeat buyers are **38%** of 2018 actives and **31%** of 2019 actives.

**Area-proportional Venn diagram** (see Appendix A below for the geometry — it's a small numerical solve, easy in `scipy.optimize`).

### 2.4 Profit by activity group
Cross-tabulate `years` against 2018 profit and 2019 profit:

| Group | 2018 profit | 2019 profit |
|---|---|---|
| Both years | ≈ $1,216,349 | ≈ $1,183,785 |
| 2018 only | ≈ $1,073,946 | — |
| 2019 only | — | ≈ $1,615,125 |
| **Total** | **≈ $2,290,295** | **≈ $2,798,910** |

**Findings:** the both-years group produces **53% of 2018 profit but only 42% of 2019 profit**, and their profit *fell* by **≈ $32,618** year on year. Growth came entirely from acquisition. Repeat customers are over-represented in profit relative to their headcount share (38%/31%) — explain that gap with the AOF × AOV × Margin decomposition applied to each of the three groups (active customers, total trans, total spend, total profit → AOF, AOV, margin per group per year).

### 2.5 Decile change analysis
Do high-value customers stay high-value?

1. Compute **profit-decile boundaries separately for 2018 and 2019** (same cumulative-profit method as §1.7b).
2. Compare them — in this dataset they come out very close. So use a **common set of cut-offs for both years**: average the two years' thresholds and round to the nearest $5, giving:

   **545, 350, 255, 195, 150, 115, 90, 65, 40**

   Decile 1 = profit > $545; decile 2 = (350, 545]; … decile 10 = ≤ $40 (including all negatives).

3. Assign every customer a 2018 decile and a 2019 decile using those cut-offs. Customers inactive in a year get profit 0 → decile 10, which is harmless because you filter by `years` before tabulating.
4. Build a 2018-decile × 2019-decile cross-tab **restricted to the `both` group** (9,871 customers).
5. Append a **`2018 only` column** (counts by 2018 decile, 16,383 customers) and a **`2019 only` row** (counts by 2019 decile, 21,984 customers).
6. Row totals as % of 26,254; column totals as % of 31,855.

**Trade-off to be explicit about:** common cut-offs mean each decile no longer holds exactly 10% of each year's profit. Year-specific cut-offs preserve the 10%-of-profit property but make the transition matrix harder to read. Analyst's call.

**Finding to expect:** heavy mass on the diagonal and just below it, plus a very large `2018 only` column concentrated in the low deciles — i.e. churn is heaviest among low-value customers, but decile 1 is not immune.

### 2.6 Up-down analysis
For the 9,871 customers active in both years, decompose profit change into its drivers.

For each customer, build four binary flags (define **Up = 2019 value ≥ 2018 value**):
- `profit_up`
- `trans_up`
- `aspt_up` — average spend per transaction (`spend/trans`)
- `margin_up` — (`profit/spend`)

Concatenate into a 4-bit pattern → 16 possible groups. Aggregate: customer count, total 2018 profit, total 2019 profit, and profit change per group. Add rows for `2018 only` (all profit lost) and `2019 only` (all profit new), then a total.

**Findings and gotchas:**
- **One customer has zero 2019 spend → undefined margin.** Drop them (analysis then covers 9,870) or force them into the Down-Down-Down-Down group.
- Only **14 of 16 groups** appear in this 1% sample. **Up-Down-Down-Down** and **Down-Up-Up-Up** are missing — but they are *logically possible*, not impossible: a loss-making customer who becomes less unprofitable can have profit Up while all three drivers are Down, and vice versa. In the full dataset there are 29 and 38 such customers respectively. **Don't hard-code 14 groups.**
- **The `≥` in the Up definition matters a lot for transactions.** Of the 6,524 customers labelled Up on transactions, **3,273 had the *same* transaction count in both years** — a third of the repeat base. Profit ties are rare (4 customers); spend ties are rare (41). **Recommendation: use three levels (Up / Same / Down) for transactions**, two for the rest.
- Largest positive contribution comes from Up-Up-Up-Up (~1,836 customers, +$221k); largest negative from Down-Down-Down-Down (~931 customers, –$170k). But the dominant swing factors overall are the 2018-only (–$1.07M) and 2019-only (+$1.62M) blocks.

---

## 3. Lens 3 — How does a cohort evolve? (Q1/2016 cohort)

**Objective:** track one acquisition cohort across its lifetime. Cohort = customers whose first-ever purchase was in Q1/2016.

**Cohort size: 2,944 customers.** (Quarterly granularity means TCBA Figures 5.1 and 5.8, which need weekly/daily data, can't be reproduced.)

### 3.1 Working dataset A — quarterly cohort summary
Filter long data to `Cohort == 'y2016 q1'`, group by `YearQuarter`: number of active cohort members, total transactions, total spend, total profit. 16 rows.

### 3.2 Revenue decomposition over time
```
Revenue_t = cohort size × %active_t × ASPAC_t
ASPAC_t   = AOF_t × AOV_t
```
Plot each component by quarter:
- **Cohort revenue** — massive drop in the quarter *after* acquisition, then slow decline with a mild Q4 seasonal bump each year.
- **% active** — `active_t / cohort_size`. Same cliff.
- **ASPAC** — `spend_t / active_t`.
- **AOF** — `trans_t / active_t`. (In the 1% sample, AOF seasonality is much weaker than in the full dataset — a sampling artefact worth noting.)
- **AOV** — `spend_t / trans_t`.

The point of the decomposition: revenue decay is driven overwhelmingly by **% active** collapsing, not by spend-per-active-buyer eroding.

### 3.3 Annual repeat-buying patterns
Build a customer × quarter transaction matrix for the cohort, then collapse to four annual binary flags:

- **2016 flag** = 1 if the customer made **more than one** transaction in 2016 (i.e. at least one *repeat* purchase beyond the acquisition purchase). *This year is special — do not use `> 0`.*
- **2017 / 2018 / 2019 flags** = 1 if any transaction in that year.

Concatenate to a 4-bit pattern → 16 groups; report count and % of cohort per pattern, sorted descending.

**Headline finding: 45% of the Q1/2016 cohort never made a second purchase by the end of 2019.** The "always on" pattern (Y-Y-Y-Y) is ~8%.

Present the table with Y/N rather than 1/0 — audiences parse it faster.

### 3.4 Time to second purchase
From the same customer × quarter matrix, build a cumulative "has made a second-ever purchase" indicator:

- Quarter 1 (acquisition quarter): `1 if trans_q1 > 1 else 0`
- Quarter t > 1: `max(indicator_{t-1}, 1 if trans_t > 0 else 0)`

Column sums ÷ cohort size → **cumulative % of cohort having made a second purchase** by end of each quarter. First differences → **% making their second purchase *in* each quarter**.

### 3.5 Quarterly repeat-buying rate *(optional; not in TCBA)*
Definition: % of customers active in period *t* who are also active in period *t+1*.

Build a customer × quarter binary activity matrix (`trans > 0`), then:
```
RBR_t = Σ (active_t × active_{t+1}) / Σ active_t
```
(i.e. dot product of adjacent columns over the sum of the earlier column.)

**Conceptual distinction:** RBR is a period-to-period measure and is blind to within-period repeat purchases. A customer who bought five times in Q1/2016 and never again shows up as a repeat buyer in §3.4 but *not* in the RBR series. Both measures are needed.

### 3.6 Working dataset B — value to date (VTD)
Group the cohort's records by `CustomerID` across all 16 quarters: total transactions, total spend, total profit. **VTD = total (undiscounted) profit over 2016–2019.** 2,944 rows.

**Findings:** VTD ranges **–$23 to $3,756**. Mean **$170**, median **$78** → **72% of cohort members are below average**. 5th pct $5.56; 10th pct $12.05; top 5% above $663.09. Just over **2% have VTD > $1,000**. Total cohort VTD ≈ **$499,821**.

**Bins:** width $25, censor at $1,000.

### 3.7 VTD decile analysis
Same construction as the **profit decile report** (§1.7b), applied to VTD: each decile = 10% of the cohort's total VTD. Decile 1 cut-off ≈ **$1,373.43** (individual VTD).

Report per decile: % of cohort, % of transactions, % of spend, % of VTD, avg spend, avg VTD, AOF, AOV, margin.

**Key finding:** value concentration is driven by **frequency, not basket size**. AOF for decile 1 is ~**25×** decile 10's; AOV is only ~**2×**. High-value customers are high-value because they *come back*.

### 3.8 Annual % active by VTD decile
Cross the VTD decile label with the annual activity flags from §3.3 (but with 2016 set to 1 for everyone — by construction every cohort member was active in their acquisition year). Report, for each decile, the % of its members active in 2016, 2017, 2018, 2019.

This is the follow-through on §3.7: the top deciles stay alive; the bottom deciles vanish after year one.

### 3.9 RFM analysis
Computed at the end of the 4-year window, on the cohort.

- **R (recency)** = index of the last quarter with a transaction, 1 (Q1/2016) … 16 (Q4/2019).
- **F (frequency)** = total transactions over the four years.
- **M (monetary value)** = **average profit per transaction** = total profit / total transactions. *(Note: this is the book's definition. Other definitions exist — state yours.)*

Bins:
| Dim | Bins |
|---|---|
| R | 1 = Q1/2016 only; 2 = Q2/2016–Q4/2017 (q2–q8); 3 = Q1/2018–Q3/2019 (q9–q15); 4 = Q4/2019 |
| F | 1 = one purchase; 2 = 2–4; 3 = 5–10; 4 = 11+ |
| M | 1 = ≤ $25; 2 = ($25, $50]; 3 = ($50, $75]; 4 = > $75 |

Cross-tab: rows = R × M, columns = F.

**Structural note:** 4×4×4 = 64 cells, but only **52 are feasible** — a customer with F = 1 made their only purchase in the acquisition quarter, so **F = 1 implies R = 1**. Twelve cells (F=1 with R ∈ {2,3,4}) are structurally empty. Don't treat them as zeros to be explained.

Bin boundaries are judgment calls. The two that are near-mandatory: a **standalone F = 1 bin**, and **standalone recency bins for the first and last periods**.

---

## 4. Lens 4 — Comparing cohorts

**Objective:** compare and contrast the performance of different acquisition cohorts, controlling for cohort size.

### Working dataset
Four **cohort × quarter** matrices (17 quarterly cohorts incl. `pre 2016`, by 16 quarters):
1. number of active customers
2. total transactions
3. total spend
4. total profit

**Cohort size** = the diagonal of matrix (1) — the acquisition quarter's active count. **Exclude the `pre 2016` row** from anything requiring cohort size.

Then derive three more matrices:
- **% active** = active / cohort size *(quarterly cohorts only)*
- **AOF** = trans / active *(all cohorts, incl. pre-2016)*
- **AOV** = spend / trans
- **Avg margin** = profit / spend

### 4.1 Cohort comparison workflow (Q3/2016 vs Q4/2016)
Cohort sizes: **Q3/2016 = 2,842**, **Q4/2016 = 6,162** — more than 2× apart, so raw comparison is meaningless.

1. **Plot raw quarterly profit** for both cohorts. Q4 dominates simply because it's bigger.
2. **Index each cohort's profit to its own acquisition-quarter profit** (= 100). This normalises for size — but it still tells you *nothing about why* one decays faster than the other.
3. **Decompose.** Plot, for each cohort, on the same axes:
   - % cohort active by quarter
   - AOF by quarter
   - AOV by quarter
   - average margin by quarter

   This is where the actual insight lives. The decomposition tells you whether a cohort underperforms because fewer of them come back, because they come back less often, because they spend less per order, or because they buy lower-margin goods. Those are four completely different problems with four completely different responses.

4. Repeat for **Q4/2016 vs Q4/2017** — a like-for-like seasonal comparison across years.

---

## 5. Lens 5 — Health of the customer base

**Objective:** firm-level view. Are we growing because the base is healthy, or because we're outrunning churn with acquisition?

### Working datasets
Annual cohorts: **pre-2016, 2016, 2017, 2018, 2019**. Derive each customer's `CohortYear` from their `Cohort` label (keep `pre y2016` as its own level; otherwise take the year part).

Four **annual cohort × calendar year** matrices (5 × 4):
1. **Number active** — count of cohort members with ≥1 transaction in that year. *(This is the awkward one: build a customer × year activity indicator first, then group by cohort year and sum.)*
2. **Total transactions**
3. **Total spend**
4. **Total profit**

Also keep a per-customer table: `CustomerID, trans_2016..trans_2019, CohortYear` — several later analyses need it.

**Validation targets — active customers by year:** 2016 = 20,673; 2017 = 21,434; 2018 = 26,254; 2019 = 31,855.

**Validation targets — total profit by year:** 2016 = $1,871,911; 2017 = $1,953,229; 2018 ≈ $2,290,343; 2019 ≈ $2,798,904.

### 5.1 Annual performance
- Bar chart: annual revenue and profit.
- **Stacked bar: annual profit by acquisition cohort.** This is the single most important picture in the audit.
- Bar chart: **new customers acquired each year** (the diagonal of matrix 1).

### 5.2 The two percentages that matter
For the profit-by-cohort stack, annotate two different ratios — they answer different questions.

**(a) Share of a year's profit from that year's new customers.**
2016: $1,193,524 / $1,871,911 = **64%** → 36% of 2016 profit came from customers acquired earlier.

**(b) Year-on-year retention of profit from existing cohorts.**
Profit in 2017 from all cohorts acquired *before* 2017 = $1,953,229 – $964,671 = $988,558.
That's **53%** of what those same cohorts delivered in 2016 ($1,871,911).

And per-cohort:
- The 2016 cohort delivered $1,193,524 in 2016 and $451,670 in 2017 → **38%** retention of profit.
- The pre-2016 cohort delivered $678,387 in 2016 and $536,888 in 2017 → **79%** retention.

**The story:** new cohorts decay fast (38% in year two); old, self-selected surviving cohorts are far stickier (79%). Growth in total profit is being bought with acquisition, while the profit contributed by each existing cohort falls roughly by half annually. Repeat the same annotation for the active-customer stack.

### 5.3 Time to second purchase, by annual cohort
Cumulative % of each annual cohort that has made a second-ever purchase by end of each year.

Logic per customer × year cell:
- 0 if the year precedes the cohort year
- if year == cohort year: `1 if trans > 1 else 0` (repeat beyond the acquisition purchase)
- if year > cohort year: `max(previous_flag, 1 if trans > 0 else 0)`

Sum by cohort year, divide by cohort size. **Exclude the `pre 2016` cohort** — its size is unknown and its "acquisition year" is outside the window.

Result is a triangular table (each cohort's series starts in its acquisition year).

### 5.4 Annual repeat-buying rate
```
RBR(cohort, x→x+1) = # cohort members active in BOTH year x and x+1
                     ÷ # cohort members active in year x
```
Build pairwise activity indicators (16/17, 17/18, 18/19) at the customer level, sum by cohort year, then divide by the corresponding column of matrix (1).

Report per cohort **and** overall (all customers, not split by cohort).

Rows: pre-2016, 2016, 2017, 2018, Overall. **There is no 2019 row** — you'd need 2020 data. Expect the pre-2016 cohort's RBR to be substantially higher than any new cohort's: survivorship, not superiority.

### 5.5 Full cohort decomposition by year
For each annual cohort × year:

| Table | Formula |
|---|---|
| **% active** | active / cohort size *(2016–2019 cohorts only; NaN for pre-2016)* |
| **Avg annual profit per active member** | profit / active |
| **Annual AOF** | trans / active |
| **Annual AOV** | spend / trans |
| **Annual avg margin** | profit / spend |

Plot each as a line chart, one line per cohort. Add a **Total** row (all customers) to each — the totals row of the last three gives you the **overall AOF, AOV and margin by year**, which is the firm-level summary of whether the business is changing shape.

**Note on plotting:** cells before a cohort exists must be **missing (NaN), not zero** — a zero will be plotted and will distort the line.

### 5.6 Quarterly version
Same three pictures, at quarterly granularity using quarterly cohorts (reuse the Lens 4 matrices):
1. Quarterly revenue and profit (column totals of the spend and profit matrices).
2. Quarterly profit **stacked by quarterly cohort** — the fine-grained version of §5.1.
3. Number of customers acquired each quarter (the diagonal).

---

## Appendix A — Area-proportional Venn diagram (Lens 2)

Excel can't draw this; matplotlib can (or use `matplotlib_venn`, though solving it yourself is trivial and gives you exact control).

Given: 2018 actives = 26,254; 2019 actives = 31,855; both = 9,871.

Set the 2018 circle radius **R = 1**. Then, requiring areas proportional to counts:

```
r = sqrt(31855 / 26254) = 1.1015
```

Required overlap area (in the same normalised units):

```
A = π × (9871 / 26254) = 1.1812
```

Solve numerically for the centre-to-centre distance **d** satisfying the standard circle-circle intersection area:

```
A(d) = r² · arccos( (d² + r² − R²) / (2dr) )
     + R² · arccos( (d² − r² + R²) / (2dR) )
     − ½ · sqrt( (d + r − R)(d − r + R)(−d + r + R)(d + r + R) )
```

Minimise `(A_target − A(d))²` over d (any 1-D root-finder / `scipy.optimize.brentq` on `A(d) − A_target`). **Solution: d = 1.1469.**

Draw: circle radius 1 centred at `(1, max(1, r))`; circle radius r centred at `(1 + d, max(1, r))`.

---

## Appendix B — Implementation checklist

- [ ] Load long CSV once; derive `Year`; keep as the single source of truth.
- [ ] Write one reusable `describe_and_bin(series, width, cutoff, neg_bin=False)` helper — six of the histograms in Lens 1/3 are the same function with different arguments.
- [ ] Write one reusable `decile_report(df, value_col, method='customer'|'value')` — Lens 1 uses it twice, Lens 3 once.
- [ ] Write one reusable `decompose(df_grouped)` returning %cust / %trans / %spend / %profit / avg spend / avg profit / AOF / AOV / margin. Used in Lens 1, 2, 3, 4, 5.
- [ ] Guard every ratio against zero denominators (`Spend == 0` → margin undefined; `NumTrans == 0` → ASPT undefined). Two customers in 2019, one in the Lens 2 both-group. Do not silently zero-fill.
- [ ] Keep pre-2016 cohort in transaction/spend/profit aggregates but **out of** every cohort-size-denominated ratio.
- [ ] Use NaN (not 0) for cohort-year cells that pre-date the cohort's existence.
- [ ] Cross-check totals at every stage: 31,855 / 60,730 / $5.84M / $2.80M for 2019; 48,238 customers in the Lens 2 frame; 2,944 in the Q1/2016 cohort.